# Latency + TTS đúng câu — bài băng tải (Arduino UNO Q)

Bổ sung cho `eval_project.ipynb` (STT / TTS / CV) và `edge_slm_benchmark.ipynb` (accuracy + RAM của SLM). Hai chỗ còn thiếu:

| Thiếu gì | Vì sao | Mục |
|---|---|---|
| **TTS có đọc đúng câu không** | Round-trip WER cũ dùng Whisper *small* đa ngữ làm "người nghe" nên WER ~0.30–0.59 là **sai số của người nghe**, không phải của TTS (chính Whisper-small đó cũng WER 0.36 trên lệnh sạch ở A2). Cần judge mạnh hơn / tiếng Anh, và đo thêm *exact-match* + *từ khoá*. Không đo ồn. | A |
| **SLM mất bao lâu để trả lời** | `edge_slm_benchmark` chỉ đo RAM + accuracy; `e2e` cũ đặt `slm_ms=0`. | B |
| **Từ lúc có lệnh đến lúc nghe thấy phản hồi** | Đo STT → NLU → SLM → TTS trong *một* tiến trình, tách đường nhanh (status/stop) và đường có SLM (why). So sánh *chờ SLM nói xong rồi mới đọc* với *đọc ngay câu đầu*. | C |

> Mọi số đo trên PC chỉ để **so sánh tương đối**; phải chạy lại đúng notebook này trên UNO Q (đặc biệt cột latency, nhiệt, `hog`).
> Cần `bench_lat.py` + `workers_lat.py` cạnh `bench_ext.py` / `workers_ext.py`.

In [1]:
%pip install -q faster-whisper vosk piper-tts jiwer soundfile librosa psutil num2words pandas matplotlib "llama-cpp-python>=0.3.0" huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys; sys.path.insert(0, ".")
from pathlib import Path
from bench_lat import *      # bx, table, flat, run_isolated (đã trỏ sang workers_lat.py), EN_* ...

CACHE = Path("model_cache")  # nơi edge_slm_benchmark đã tải các GGUF
GGUF = {
    "SmolLM2-135M":  CACHE / "SmolLM2-135M-Instruct-Q4_K_M.gguf",
    "Qwen2.5-0.5B":  CACHE / "qwen2.5-0.5b-instruct-q4_k_m.gguf",
    "TinyLlama-1.1B": CACHE / "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    "Phi-3.5-mini":  CACHE / "Phi-3.5-mini-instruct-Q4_K_M.gguf",
}
GGUF = {k: str(v) for k, v in GGUF.items() if v.exists()}
print("GGUF tìm thấy:", list(GGUF))

GGUF tìm thấy: ['SmolLM2-135M', 'Qwen2.5-0.5B', 'TinyLlama-1.1B', 'Phi-3.5-mini']


## A. TTS — có đọc đúng câu không? (không ồn)

Mỗi câu: Piper đọc → *judge* STT nghe lại → so với câu tham chiếu. Chỉ số:
- `exact_*`: tỉ lệ câu nghe lại **khớp hoàn toàn** (sau bỏ dấu câu/hoa thường; chữ số → chữ).
- `key_recall_*`: tỉ lệ **từ khoá/số bắt buộc** nghe được (vd "jammed", "three point two"). Với cảnh báo an toàn, đây quan trọng hơn WER.
- `rt_wer_*`, `rt_cer_*`: round-trip WER/CER.
- Ba câu có số được thử **hai dạng**: chữ số thô ("0.8") và đã viết bằng chữ — nếu dạng thô sai mà dạng chữ đúng thì phải chuẩn hoá số trước khi đưa vào TTS.

**Cách đọc:** chỉ tin kết quả khi judge mạnh. Nếu `small.en` và `medium.en` cho kết quả khác nhau nhiều → còn bị giới hạn bởi judge; lấy số của judge mạnh nhất. Nghe thử thêm các file `.wav` trong `tts_out_en/` ở các câu `mismatch_*`.

In [3]:
VOICES = [("piper en lessac-medium", "en_US-lessac-medium"),
          ("piper en amy-medium",    "en_US-amy-medium"),
          ("piper en ryan-medium",   "en_US-ryan-medium")]      # 404 -> xem huggingface.co/rhasspy/piper-voices
JUDGES = ["small.en", "medium.en"]                               # medium.en ~1.5 GB tải về; bỏ nếu máy yếu
for name, voice in VOICES:
    bx("tts_correct", name, "tts_correct",
       {"voice": voice, "out_dir": f"tts_out_en/{voice}", "sentences": EN_TTS_SENTENCES,
        "judges": JUDGES, "language": "en"})
dt = table("tts_correct")
cols = ["model", "ram_total_peak_mb", "ttfa_ms_p50", "synth_ms_p50", "rtf", "audio_s_mean"] + \
       [f"{m}_{j.replace('.', '_')}" for j in JUDGES for m in ("exact", "key_recall", "rt_wer")]
dt[[c for c in cols if c in dt.columns]]

[done] piper en lessac-medium: ram_peak_mb=207.3, ram_total_peak_mb=229.3, rtf=0.03
     {'ttfa_ms_p50': 77.0, 'ttfa_ms_p95': 98.0, 'synth_ms_p50': 77.0, 'synth_ms_p95': 98.0, 'audio_s_mean': 2.52, 'rtf': 0.03, 'n_sentences': 13, 'wav_dir': 'tts_out_en\\en_US-lessac-medium', 'onnx_mb': 60.3, 'runtime': 'onnxruntime(piper)', 'rt_wer_small_en': 0.0, 'rt_cer_small_en': 0.0, 'exact_small_en': 1.0, 'key_recall_small_en': 1.0, 'rt_wer_medium_en': 0.0202, 'rt_cer_medium_en': 0.0106, 'exact_medium_en': 0.8462, 'key_recall_medium_en': 0.9615}
[done] piper en amy-medium: ram_peak_mb=223.6, ram_total_peak_mb=245.4, rtf=0.03
     {'ttfa_ms_p50': 107.0, 'ttfa_ms_p95': 114.6, 'synth_ms_p50': 107.0, 'synth_ms_p95': 114.6, 'audio_s_mean': 3.17, 'rtf': 0.03, 'n_sentences': 13, 'wav_dir': 'tts_out_en\\en_US-amy-medium', 'onnx_mb': 60.3, 'runtime': 'onnxruntime(piper)', 'rt_wer_small_en': 0.0, 'rt_cer_small_en': 0.0, 'exact_small_en': 1.0, 'key_recall_small_en': 1.0, 'rt_wer_medium_en': 0.0, 'rt_cer_medi

,model,ram_total_peak_mb,ttfa_ms_p50,synth_ms_p50,rtf,audio_s_mean,exact_small_en,key_recall_small_en,rt_wer_small_en,exact_medium_en,key_recall_medium_en,rt_wer_medium_en
0,piper en amy-medium,245.4,107.0,107.0,0.030,3.17,1.0000,1.0,0.0000,1.0000,1.0000,0.0000
1,piper en lessac-medium,229.3,77.0,77.0,0.030,2.52,1.0000,1.0,0.0000,0.8462,0.9615,0.0202
2,piper en ryan-medium,225.5,83.9,83.9,0.031,2.43,0.9231,1.0,0.0101,0.8462,0.9808,0.0202


In [4]:
# Câu bị nghe sai (judge mạnh nhất) — nghe lại file wav tương ứng để biết lỗi do TTS hay do judge
best = "medium_en" if "medium.en" in JUDGES else JUDGES[-1].replace(".", "_")
for _, r in dt.iterrows():
    print(r["model"], "->", f"exact={r.get('exact_' + best)}", f"key_recall={r.get('key_recall_' + best)}")
    for s in (r.get("mismatch_" + best) or []):
        print("    ", s)

piper en amy-medium -> exact=1.0 key_recall=1.0
piper en lessac-medium -> exact=0.8462 key_recall=0.9615
     Motor current is three point two amps.  =>  Motor current is 3.2A.
     Motor current is three point two amps.  =>  Motor current is 3.2A.
piper en ryan-medium -> exact=0.8462 key_recall=0.9808
     Belt speed is zero point eight meters per second.  =>  The belt speed is 0.8 meters per second.
     Motor current is three point two amps.  =>  Motor current is 3.2A.


*(Tuỳ chọn — tiếng Việt, chỉ để dành nếu thầy yêu cầu.)* Cùng worker, đổi `language="vi"`, `judges=["small", "medium"]`, câu `VI_TTS_SENTENCES` và giọng `vi_VN-vais1000-medium`. Số chữ số → chữ: `num2words` hỗ trợ `vi`, nhưng nên viết số bằng chữ như hiện tại.

## B. SLM — mất bao lâu để trả lời?

`gguf` chạy **trong tiến trình** bằng llama-cpp-python (không qua HTTP, không bị cộng độ trễ mạng). Prompt = system + snapshot cảm biến (JSON) + câu hỏi, giống `Multimodal Context Builder`; trả lời ≤ 64 token, temperature 0.
- `load_s`: thời gian nạp model (quan trọng nếu nạp SLM theo yêu cầu như báo cáo đề xuất). `first_call_ttft_ms`: lần gọi đầu sau khi nạp (nguội).
- `ttft_ms_*`: tới token đầu (= prefill + token đầu). `decode_tps_*`: token/giây khi sinh. `total_ms_*`: tới khi trả lời xong.
- `mode`: **cold** = xoá cache mỗi câu (xấu nhất) · **prefix** = giữ cache nên system prompt chung không phải prefill lại.
- `pad`: số token telemetry thêm vào prompt (0 / 512) → xem TTFT tăng thế nào theo ngữ cảnh.
- `hog2` mô phỏng CPU bị camera/STT chiếm; `cpus=[0,1,2,3]` ghim 4 lõi (chỉ tương đối — lõi A53 chậm hơn lõi PC nhiều).

In [5]:
for name, path in GGUF.items():
    for threads, hogs in [(4, 0), (4, 2)]:
        bx("slm_latency", f"{name} t{threads} hog{hogs}", "slm_latency",
           {"gguf": path, "threads": threads, "hog_procs": hogs, "cpus": [0, 1, 2, 3] if hogs else None,
            "pads": [0, 512], "modes": ["cold", "prefix"], "reps": 4, "max_tokens": 64},
           force=False)
ds = table("slm_latency")
cols = ["model", "ram_total_peak_mb", "load_s", "first_call_ttft_ms", "ttft_ms_p50_cold_pad0", "ttft_ms_p50_prefix_pad0",
        "ttft_ms_p50_cold_pad512", "decode_tps_p50_cold_pad0", "total_ms_p50_cold_pad0", "total_ms_p95_cold_pad0",
        "out_tokens_cold_pad0"]
ds[[c for c in cols if c in ds.columns]]

[done] SmolLM2-135M t4 hog0: ram_peak_mb=229.2, ram_total_peak_mb=229.2
     {'ttft_ms_p50': 66.4, 'ttft_ms_p95': 114.0, 'decode_tps_p50': 263.5, 'total_ms_p50': 116.9, 'total_ms_p95': 252.9, 'headline_condition': 'cold_pad0', 'ttft_ms_p50_cold_pad0': 66.4, 'ttft_ms_p95_cold_pad0': 114.0, 'decode_tps_p50_cold_pad0': 263.5, 'prefill_tps_approx_cold_pad0': 1401.0, 'total_ms_p50_cold_pad0': 116.9, 'total_ms_p95_cold_pad0': 252.9, 'prompt_tokens_cold_pad0': 96.0, 'out_tokens_cold_pad0': 14.0, 'ttft_ms_p50_prefix_pad0': 35.9, 'ttft_ms_p95_prefix_pad0': 43.1, 'decode_tps_p50_prefix_pad0': 263.0, 'prefill_tps_approx_prefix_pad0': 2651.2, 'total_ms_p50_prefix_pad0': 81.2, 'total_ms_p95_prefix_pad0': 228.4, 'prompt_tokens_prefix_pad0': 96.0, 'out_tokens_prefix_pad0': 14.0, 'ttft_ms_p50_cold_pad512': 365.2, 'ttft_ms_p95_cold_pad512': 458.7, 'decode_tps_p50_cold_pad512': 203.2, 'prefill_tps_approx_cold_pad512': 1693.3, 'total_ms_p50_cold_pad512': 494.6, 'total_ms_p95_cold_pad512': 623.6, 'prompt_

,model,ram_total_peak_mb,load_s,first_call_ttft_ms,ttft_ms_p50_cold_pad0,ttft_ms_p50_prefix_pad0,ttft_ms_p50_cold_pad512,decode_tps_p50_cold_pad0,total_ms_p50_cold_pad0,total_ms_p95_cold_pad0,out_tokens_cold_pad0
0,Phi-3.5-mini t4 hog0,4368.6,1.77,1356.0,1128.2,548.1,5754.4,17.9,3673.0,4619.5,47.0
1,Phi-3.5-mini t4 hog2,4368.7,1.26,4146.0,3819.7,2084.5,20979.8,10.1,8394.2,9920.9,47.0
2,Qwen2.5-0.5B t4 hog0,602.0,0.34,182.0,145.8,72.2,882.2,96.2,318.1,396.3,14.0
3,Qwen2.5-0.5B t4 hog2,602.0,0.31,538.0,509.2,269.5,2990.5,43.9,870.9,1166.9,14.0
4,SmolLM2-135M t4 hog0,229.2,0.26,86.0,66.4,35.9,365.2,263.5,116.9,252.9,14.0
5,SmolLM2-135M t4 hog2,229.5,0.13,197.0,288.9,153.8,1402.0,93.8,410.0,823.8,14.0
6,TinyLlama-1.1B t4 hog0,1207.8,0.59,359.0,230.2,121.6,1151.3,72.0,1089.6,1122.3,64.0
7,TinyLlama-1.1B t4 hog2,1207.7,0.53,859.0,940.0,514.0,4654.2,37.1,2615.2,2698.0,64.0


In [6]:
# Câu trả lời mẫu của từng model (kiểm tra nhanh: có bám dữ liệu cảm biến không — accuracy thật nằm ở bộ safety/grounding riêng)
for _, r in ds.iterrows():
    print(r["model"], "->", r.get("sample_answer"))

Phi-3.5-mini t4 hog0 -> The sensor data does not explicitly state a reason for the conveyor stopping.

However, based on the available information, potential reasons could include a programmed stop or a safety feature trigge
Phi-3.5-mini t4 hog2 -> The sensor data does not explicitly state a reason for the conveyor stopping.

However, based on the available information, potential reasons could include a programmed stop or a safety feature trigge
Qwen2.5-0.5B t4 hog0 -> The conveyor stopped due to a motor current of 3.1 A, which is too high for the motor to handle.
Qwen2.5-0.5B t4 hog2 -> The conveyor stopped due to a motor current of 3.1 A, which is too high for the motor to handle.
SmolLM2-135M t4 hog0 -> The conveyor stopped due to a malfunction in the motor's current flow.
SmolLM2-135M t4 hog2 -> The conveyor stopped due to a malfunction in the motor's current flow.
TinyLlama-1.1B t4 hog0 -> The sensor data provided in the given sensor snapshot indicates that the conveyor belt stopp

## C. Từ lệnh giọng nói đến phản hồi (STT → NLU → SLM → TTS, một tiến trình)

Hai đường:
- **Đường nhanh** (status, speed, temp, count, **stop**, restart-check): STT → NLU → câu mẫu → TTS. *Không đi qua SLM.*
- **Đường SLM** (`why`): thêm SLM sinh giải thích. Đo hai cách phát: `wait` = chờ SLM nói xong rồi mới đọc · `stream` = đọc ngay câu đầu hoàn chỉnh (ước lượng tuần tự, hơi lạc quan vì thật sự TTS và SLM sẽ tranh CPU).

Mọi mốc tính **từ lúc audio lệnh đã đủ** (chưa gồm thời gian người nói và độ trễ endpointing của VAD) — dùng `vad_endpoint_ms` để cộng thêm giá trị giả định. RAM là **RAM chung thật** của STT + TTS + SLM nằm cùng lúc.
Lệnh *dừng vật lý* **không** được phụ thuộc đường này: STM32 dừng từ cảm biến/E-stop độc lập (đo riêng ở bộ test "Linux chết" trên board).

In [7]:
DATA_CMD = make_en_commands()           # lệnh tiếng Anh tổng hợp bằng Piper — đủ cho latency (không dùng để kết luận độ chính xác STT)
GRAMMAR = sorted({p.lower() for ps in EN_VARIANTS.values() for p in ps}) + ["[unk]"]
STT_CFGS = [
    ("fw tiny.en",  {"engine": "fw", "size": "tiny.en",  "language": "en"}),
    ("fw base.en",  {"engine": "fw", "size": "base.en",  "language": "en"}),
    ("fw small.en", {"engine": "fw", "size": "small.en", "language": "en"}),
    ("vosk en-small +grammar", {"engine": "vosk", "model": "vosk-model-small-en-us-0.15", "grammar": GRAMMAR}),
]
SLM_FOR_E2E = GGUF.get("Qwen2.5-0.5B") or next(iter(GGUF.values()), None)   # None -> đường 'why' dùng câu mẫu
for name, cfg in STT_CFGS:
    for hogs in [0, 2]:
        bx("e2e_full", f"{name} | piper lessac | slm | hog{hogs}", "e2e_full",
           {**cfg, "data_dir": DATA_CMD, "voice": "en_US-lessac-medium", "gguf": SLM_FOR_E2E, "nlu": "en",
            "responses": EN_RESPONSES, "repeat": 4, "hog_procs": hogs, "cpus": [0, 1, 2, 3] if hogs else None,
            "vad_endpoint_ms": 500, "threads": 4})
de = table("e2e_full")
cols = ["model", "ram_total_peak_mb", "intent_acc", "fast_first_audio_ms_p50", "fast_first_audio_ms_p95", "stop_first_audio_ms_p95",
        "slm_ttft_ms_p50", "slm_total_ms_p50", "slm_first_audio_wait_ms_p50", "slm_first_audio_stream_ms_p50",
        "slm_first_audio_stream_ms_p95", "slm_load_s"]
de[[c for c in cols if c in de.columns]]

Đã sinh 63 lệnh tổng hợp tại data\en_commands
[done] fw tiny.en | piper lessac | slm | hog0: ram_peak_mb=1011.1, ram_total_peak_mb=1051.6
     {'n_fast': 216, 'n_slm': 36, 'intent_acc': 0.9683, 'fast_stt_ms_p50': 183.4, 'fast_stt_ms_p95': 194.7, 'fast_tts_ttfa_ms_p50': 67.7, 'fast_first_audio_ms_p50': 245.7, 'fast_first_audio_ms_p95': 271.6, 'fast_first_audio_ms_max': 372.6, 'fast_reply_end_ms_p50': 2625.2, 'stop_first_audio_ms_p95': 246.0, 'n_stop': 36, 'slm_stt_ms_p50': 184.8, 'slm_ttft_ms_p50': 160.9, 'slm_total_ms_p50': 303.6, 'slm_total_ms_p95': 431.7, 'slm_tokens_p50': 14.0, 'slm_first_audio_wait_ms_p50': 582.1, 'slm_first_audio_wait_ms_p95': 751.0, 'slm_first_audio_stream_ms_p50': 588.8, 'slm_first_audio_stream_ms_p95': 758.7, 'assumed_vad_endpoint_ms': 500, 'fast_first_audio_incl_vad_ms_p50': 745.7, 'slm_first_audio_stream_incl_vad_ms_p50': 1088.8, 'slm_load_s': 0.32, 'hog_procs': 0, 'cpus': None, 'n_samples': 252, 'runtime': 'stt+nlu+slm+piper'}
[done] fw tiny.en | piper lessa

,model,ram_total_peak_mb,intent_acc,fast_first_audio_ms_p50,fast_first_audio_ms_p95,stop_first_audio_ms_p95,slm_ttft_ms_p50,slm_total_ms_p50,slm_first_audio_wait_ms_p50,slm_first_audio_stream_ms_p50,slm_first_audio_stream_ms_p95,slm_load_s
0,fw base.en | piper lessac | slm | hog0,1172.2,0.9841,399.5,474.9,478.6,172.4,322.2,774.2,785.9,1071.4,0.29
1,fw base.en | piper lessac | slm | hog2,1206.4,0.9841,1070.0,1298.8,1171.6,614.6,956.9,2179.5,2208.1,2992.7,0.31
2,fw small.en | piper lessac | slm | hog0,1332.6,1.0000,1297.5,1393.6,1337.0,231.8,419.2,1776.5,1783.3,2055.0,0.38
3,fw small.en | piper lessac | slm | hog2,1356.6,1.0000,2071.8,2250.6,2074.5,559.3,914.7,3192.5,3191.1,3755.4,0.33
4,fw tiny.en | piper lessac | slm | hog0,1051.6,0.9683,245.7,271.6,246.0,160.9,303.6,582.1,588.8,758.7,0.32
5,fw tiny.en | piper lessac | slm | hog2,1163.7,0.9683,574.4,660.5,528.6,540.2,849.3,1549.1,1547.5,1963.4,0.29
6,vosk en-small +grammar | piper lessac | slm | ...,1079.5,1.0000,121.8,156.4,106.4,216.4,403.1,576.6,586.6,814.2,0.39
7,vosk en-small +grammar | piper lessac | slm | ...,1058.3,1.0000,327.1,413.1,279.3,587.5,935.9,1381.1,1402.8,1870.5,0.39


In [8]:
# Đuôi dài: p95 >> p50 thường do vài mẫu STT chạy rất lâu (Whisper lặp từ trên audio ngắn). In 3 mẫu chậm nhất để tìm nguyên nhân.
for _, r in de.iterrows():
    print(r["model"]); [print("    ", s) for s in (r.get("slowest_stt") or [])]

fw base.en | piper lessac | slm | hog0
     550 ms | ref='What is the belt speed' | hyp='What is the belt speed?'
     508 ms | ref='What is the conveyor status' | hyp='What is the conveyor status?'
     474 ms | ref='Is the motor too hot' | hyp='Is the motor too hot?'
fw base.en | piper lessac | slm | hog2
     1129 ms | ref='Is the conveyor running' | hyp='Is the conveyor running?'
     996 ms | ref='Is the conveyor running' | hyp='Is the conveyor running?'
     969 ms | ref='Count the products today' | hyp='Count the products today.'
fw small.en | piper lessac | slm | hog0
     1340 ms | ref='What is the conveyor status' | hyp='What is the conveyor status?'
     1336 ms | ref='How fast is the belt moving' | hyp='How fast is the belt moving?'
     1332 ms | ref='Tell me the reason it stopped' | hyp='Tell me the reason it stopped.'
fw small.en | piper lessac | slm | hog2
     2179 ms | ref='Tell me the reason it stopped' | hyp='Tell me the reason it stopped.'
     2178 ms | ref='What 

## D. Ngân sách thời gian so với mục tiêu

Ngưỡng dưới đây là **mục tiêu tự đặt**, không phải chuẩn — chỉnh theo yêu cầu của nhóm rồi ghi vào báo cáo cùng lý do.

In [9]:
TARGET_FAST_P95_MS   = 2000   # hỏi trạng thái -> nghe câu trả lời
TARGET_STOP_P95_MS   = 1500   # lệnh dừng bằng giọng -> phản hồi xác nhận (dừng vật lý do STM32, không tính ở đây)
TARGET_WHY_STREAM_MS = 5000   # 'why' -> nghe câu giải thích đầu tiên (p95)
rows = []
for _, r in de.iterrows():
    f, s, w = r.get("fast_first_audio_ms_p95"), r.get("stop_first_audio_ms_p95"), r.get("slm_first_audio_stream_ms_p95")
    rows.append({"config": r["model"], "ram_mb": r.get("ram_total_peak_mb"), "fast_p95": f, "stop_p95": s, "why_stream_p95": w,
                 "PASS": (f is not None and f <= TARGET_FAST_P95_MS) and (s is None or s <= TARGET_STOP_P95_MS)
                         and (w is None or w <= TARGET_WHY_STREAM_MS)})
import pandas as pd
pd.DataFrame(rows)

,config,ram_mb,fast_p95,stop_p95,why_stream_p95,PASS
0,fw base.en | piper lessac | slm | hog0,1172.2,474.9,478.6,1071.4,True
1,fw base.en | piper lessac | slm | hog2,1206.4,1298.8,1171.6,2992.7,True
2,fw small.en | piper lessac | slm | hog0,1332.6,1393.6,1337.0,2055.0,True
3,fw small.en | piper lessac | slm | hog2,1356.6,2250.6,2074.5,3755.4,False
4,fw tiny.en | piper lessac | slm | hog0,1051.6,271.6,246.0,758.7,True
5,fw tiny.en | piper lessac | slm | hog2,1163.7,660.5,528.6,1963.4,True
6,vosk en-small +grammar | piper lessac | slm | ...,1079.5,156.4,106.4,814.2,True
7,vosk en-small +grammar | piper lessac | slm | ...,1058.3,413.1,279.3,1870.5,True


## Cách dùng kết quả trong báo cáo
1. **TTS:** báo `exact_*` và `key_recall_*` của judge mạnh nhất, kèm danh sách câu sai; nêu rõ judge và rằng round-trip chỉ là proxy của độ dễ nghe (không thay thế nghe trực tiếp).
2. **SLM:** lập bảng `load_s`, `ttft`, `decode_tps`, `total_ms` theo model, ghi rõ `cold` hay `prefix`, số luồng, `pad`, `max_tokens`, và máy đo. Kết hợp RAM + accuracy từ `edge_slm_benchmark` thành biểu đồ ba chiều (accuracy – RAM – latency).
3. **E2E:** báo p50 **và** p95, tách đường nhanh/đường SLM; nêu mức tiết kiệm của `stream` so với `wait`. Phần cứng khác thì con số tuyệt đối đổi — chỉ chốt trên UNO Q.
4. **Giới hạn cần nêu thẳng:** audio lệnh là tổng hợp (không phản ánh giọng thật/ồn); đo trên PC; `stream` là ước lượng tuần tự; chưa gồm độ trễ VAD/endpointing nếu không đặt `vad_endpoint_ms`.